## 1. Import Libraries

In [17]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Machine Learning (tradicional)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import adjusted_rand_score, silhouette_score, homogeneity_score, completeness_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.manifold import TSNE

# OpenCV para processamento de imagem (features do paper SVHN)
import cv2
from scipy import ndimage
from skimage.feature import local_binary_pattern

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 2. Load Dataset

In [13]:
# Load training data from CSV
print("Loading training data from CSV...")
train_data = pd.read_csv('SVHN_train.csv')

# Separate features and labels
X_train_full = train_data.iloc[:, :-1].values.astype('float32')
y_train_full = train_data.iloc[:, -1].values


Loading training data from CSV...


## 3. Data Preprocessing Functions

In [25]:
def reshape_images(X):
    """Reshape flat pixel data to 32x32x3 images using row-major interpretation"""
    return X.reshape(-1, 32, 32, 3)

def simple_preprocessing(X, y, test_size=0.2, sample=True, sample_size=0.1):
    """Basic preprocessing: normalize and split data"""
    # Reshape to images
    X_images = reshape_images(X)
    
    # Normalize to [0, 1]
    X_norm = X_images.astype('float32') / 255.0
    
    # Stratified split to maintain class distribution
    if sample:
        # Sample a fraction of the data for quicker experiments
        X_sample, _, y_sample, _ = train_test_split(
            X_norm, y, test_size=1 - sample_size, stratify=y, random_state=42
        )
        X_norm, y = X_sample, y_sample

    X_train, X_val, y_train, y_val = train_test_split(
        X_norm, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Compute class weights for imbalanced data
    class_weights = compute_class_weight(
        'balanced', classes=np.unique(y), y=y
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    return X_train, X_val, y_train, y_val, class_weight_dict


## 4. Train and Evaluation method

In [ ]:
# classifiers funciontion
def create_classifier_models():
    """Create various classifier models for comparison"""
    models = {
        #'KNN': KNeighborsClassifier(n_neighbors=10),
        #'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
        #'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Neural Network': MLPClassifier(hidden_layer_sizes=(1024, 256), random_state=42, max_iter=500)
    }
    return models

def train_and_evaluate_classifiers(X_train, y_train, X_val, y_val, class_weights=None):
    """Train multiple classifiers and return results"""
    
    models = create_classifier_models()
    results = {}
    
    for name, model in models.items():
        print(f"Training {name}...")
        
        # Handle class weights for models that support it
        if hasattr(model, 'class_weight') and class_weights is not None:
            model.set_params(class_weight=class_weights)
        
        # Train model
        model.fit(X_train, y_train)
        
        # Evaluate
        train_acc = model.score(X_train, y_train)
        val_acc = model.score(X_val, y_val)
        
        results[name] = {
            'model': model,
            'train_acc': train_acc,
            'val_acc': val_acc
        }
        
        print(f"{name} - Train: {train_acc:.4f}, Val: {val_acc:.4f}")
    
    return results

In [ ]:
def extract_features(X, method='flatten'):
    """
    Extrai features das imagens usando diferentes métodos.
    
    Args:
        X: Array de imagens (n_samples, 32, 32, 3)
        method: 'flatten', 'statistical', 'svhn_enhanced'
    
    Returns:
        features: Array de features extraídas
    """
    if method == 'flatten':
        # Simplesmente achata as imagens (baseline)
        return X.reshape(X.shape[0], -1)
    
    elif method == 'statistical':
        # Features estatísticas por canal
        features = []
        for img in X:
            img_features = []
            # Para cada canal RGB
            for channel in range(3):
                ch = img[:, :, channel]
                img_features.extend([
                    np.mean(ch), np.std(ch), np.min(ch), np.max(ch),
                    np.median(ch), np.percentile(ch, 25), np.percentile(ch, 75)
                ])
                
                # Região central (mais importante baseado na EDA)
                center = ch[8:24, 8:24]
                img_features.extend([np.mean(center), np.std(center)])
            
            features.append(img_features)
            
        return np.array(features)
    else:
        raise ValueError(f"Método '{method}' não reconhecido")

print("Função extract_features atualizada com método SVHN!")

Função extract_features atualizada com método SVHN!


## 7. Evaluation Functions

In [ ]:

def plot_classifier_comparison(results, title="Classifier Comparison"):
    """Plot classifier performance comparison"""
    
    models = list(results.keys())
    train_accs = [results[model]['train_acc'] for model in models]
    val_accs = [results[model]['val_acc'] for model in models]
    
    x = np.arange(len(models))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, train_accs, width, label='Training Accuracy', alpha=0.8)
    bars2 = ax.bar(x + width/2, val_accs, width, label='Validation Accuracy', alpha=0.8)
    
    ax.set_xlabel('Models')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 8. Experiment 1: K-Means + Classifiers with Flattened Features (Baseline)

In [24]:
print("=== EXPERIMENT 1: SIMPLE PREPROCESSING (BASELINE) ===")

# Apply simple preprocessing
X_train, X_val, y_train, y_val, class_weights = simple_preprocessing(
    X_train_full, y_train_full
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Class weights: {class_weights}")

# Extract features (flatten pixels)
print("\nExtracting features (flattened pixels)...")
X_train_features = extract_features(X_train, method='flatten')
X_val_features = extract_features(X_val, method='flatten')

print(f"Training features shape: {X_train_features.shape}")
print(f"Validation features shape: {X_val_features.shape}")

# Apply scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_val_scaled = scaler.transform(X_val_features)

# Train classifiers
print("\nTraining classifiers...")
classifier_results_baseline = train_and_evaluate_classifiers(
    X_train_scaled, y_train, X_val_scaled, y_val, class_weights
)

=== EXPERIMENT 1: SIMPLE PREPROCESSING (BASELINE) ===
Training set shape: (5956, 32, 32, 3)
Validation set shape: (1490, 32, 32, 3)
Class weights: {0: np.float64(1.4686390532544378), 1: np.float64(0.5280851063829787), 2: np.float64(0.6678026905829596), 3: np.float64(0.8698598130841122), 4: np.float64(0.9967871485943776), 5: np.float64(1.0854227405247814), 6: np.float64(1.3040280210157618), 7: np.float64(1.2860103626943005), 8: np.float64(1.4686390532544378), 9: np.float64(1.591025641025641)}

Extracting features (flattened pixels)...
Training features shape: (5956, 3072)
Validation features shape: (1490, 3072)

Training classifiers...

Training MLP...
Training set shape: (5956, 32, 32, 3)
Validation set shape: (1490, 32, 32, 3)
Class weights: {0: np.float64(1.4686390532544378), 1: np.float64(0.5280851063829787), 2: np.float64(0.6678026905829596), 3: np.float64(0.8698598130841122), 4: np.float64(0.9967871485943776), 5: np.float64(1.0854227405247814), 6: np.float64(1.3040280210157618), 7

## 9. Preprocessing Inteligente baseado na Análise de Variância

In [29]:
def per_channel_standardization(X):
    """
    Normaliza cada canal RGB individualmente para cada imagem.
    Remove viés de iluminação e melhora contraste local.
    """
    X_normalized = np.zeros_like(X, dtype=np.float32)
    
    for i in range(X.shape[0]):  # Para cada imagem
        for channel in range(3):  # Para cada canal RGB
            ch_data = X[i, :, :, channel]
            mean_ch = np.mean(ch_data)
            std_ch = np.std(ch_data)
            
            if std_ch > 1e-8:  # Evita divisão por zero
                X_normalized[i, :, :, channel] = (ch_data - mean_ch) / std_ch
            else:
                X_normalized[i, :, :, channel] = ch_data - mean_ch
    
    return X_normalized

def intelligent_crop_variance_based(X, variance_threshold=3.0):
    """
    Remove colunas com baixo poder discriminativo baseado na análise de variância da EDA.
    """
    print(f"Aplicando crop inteligente com threshold {variance_threshold}...")
    
    # Baseado na observação da EDA: colunas à direita (26-31) têm baixa variância discriminativa
    if variance_threshold == 'visual':
        # Método baseado na observação visual da EDA
        X_cropped = X[:, :, :26, :]  # Manter primeiras 26 colunas
        removed_cols = list(range(26, 32))
    else:
        # Método heurístico baseado na análise de variância
        # Threshold 3.0 aproximadamente corresponde à coluna 26
        crop_point = max(20, min(30, int(32 - (10 - variance_threshold))))
        X_cropped = X[:, :, :crop_point, :]
        removed_cols = list(range(crop_point, 32))
    
    print(f"Dimensões originais: {X.shape}")
    print(f"Dimensões após crop: {X_cropped.shape}")
    print(f"Colunas removidas: {len(removed_cols)} - {removed_cols}")
    
    return X_cropped, removed_cols

def manual_crop_by_pixel(X, crop_col_start=26):
    """
    Crop manual baseado na contagem de pixels do usuário.
    """
    print(f"Aplicando crop manual a partir da coluna {crop_col_start}...")
    
    X_cropped = X[:, :, :crop_col_start, :]
    removed_cols = list(range(crop_col_start, X.shape[2]))
    
    print(f"Dimensões originais: {X.shape}")
    print(f"Dimensões após crop: {X_cropped.shape}")
    print(f"Colunas removidas: {len(removed_cols)} - {removed_cols}")
    
    return X_cropped, removed_cols

def enhanced_preprocessing_pipeline(X_train_full, y_train_full, method='variance', crop_param=3.0):
    """
    Pipeline completo de preprocessing inteligente.
    
    Args:
        method: 'variance' ou 'manual'
        crop_param: threshold para variance ou coluna para manual
    """
    
    print(f"=== PREPROCESSING INTELIGENTE - MÉTODO {method.upper()} ===")
    
    # 1. Reshape para imagens e normalização básica
    X_images = X_train_full.reshape(-1, 32, 32, 3).astype('float32') / 255.0
    
    # 2. Crop inteligente
    if method == 'variance':
        X_cropped, removed_cols = intelligent_crop_variance_based(X_images, crop_param)
    else:  # manual
        X_cropped, removed_cols = manual_crop_by_pixel(X_images, int(crop_param))
    
    # 3. Normalização per-channel
    print("\nAplicando normalização per-channel...")
    X_normalized = per_channel_standardization(X_cropped)
    
    # 4. Split estratificado
    X_train, X_val, y_train, y_val = train_test_split(
        X_normalized, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )
    
    # 5. Flatten para ML tradicional
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    
    # 6. Scaling final
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_val_scaled = scaler.transform(X_val_flat)
    
    # 7. Calcular class weights
    class_weights = compute_class_weight(
        'balanced', classes=np.unique(y_train_full), y=y_train_full
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    print(f"\nDimensões finais:")
    print(f"Training: {X_train_scaled.shape}")
    print(f"Validation: {X_val_scaled.shape}")
    print(f"Redução de features: {((3072 - X_train_scaled.shape[1]) / 3072 * 100):.1f}%")
    
    return X_train_scaled, X_val_scaled, y_train, y_val, class_weight_dict, scaler, removed_cols

print("Funções de preprocessing inteligente carregadas!")

Funções de preprocessing inteligente carregadas!


## 10. Experimento: Método baseado na Variância (EDA)

In [30]:
print("=== EXPERIMENTO 1: CROP BASEADO NA ANÁLISE DE VARIÂNCIA ===\n")

# Testar método baseado na variância (threshold 3.0)
X_train_var, X_val_var, y_train_var, y_val_var, class_weights_var, scaler_var, removed_cols_var = enhanced_preprocessing_pipeline(
    X_train_full, y_train_full, method='variance', crop_param=3.0
)

# Treinar MLP otimizado
print("\nTreinando MLP com features reduzidas...")
mlp_variance = MLPClassifier(
    hidden_layer_sizes=(1024, 256),
    random_state=42,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

mlp_variance.fit(X_train_var, y_train_var)

# Avaliar
train_acc_var = mlp_variance.score(X_train_var, y_train_var)
val_acc_var = mlp_variance.score(X_val_var, y_val_var)

print(f"\n🏆 RESULTADOS - MÉTODO VARIÂNCIA:")
print(f"Training Accuracy: {train_acc_var:.4f} ({train_acc_var*100:.2f}%)")
print(f"Validation Accuracy: {val_acc_var:.4f} ({val_acc_var*100:.2f}%)")
print(f"Overfitting Gap: {(train_acc_var - val_acc_var)*100:.2f}%")
print(f"Features removidas: {len(removed_cols_var)} colunas {removed_cols_var}")

=== EXPERIMENTO 1: CROP BASEADO NA ANÁLISE DE VARIÂNCIA ===

=== PREPROCESSING INTELIGENTE - MÉTODO VARIANCE ===
Aplicando crop inteligente com threshold 3.0...
Dimensões originais: (37233, 32, 32, 3)
Dimensões após crop: (37233, 32, 25, 3)
Colunas removidas: 7 - [25, 26, 27, 28, 29, 30, 31]

Aplicando normalização per-channel...
Aplicando crop inteligente com threshold 3.0...
Dimensões originais: (37233, 32, 32, 3)
Dimensões após crop: (37233, 32, 25, 3)
Colunas removidas: 7 - [25, 26, 27, 28, 29, 30, 31]

Aplicando normalização per-channel...

Dimensões finais:
Training: (29786, 2400)
Validation: (7447, 2400)
Redução de features: 21.9%

Treinando MLP com features reduzidas...

Dimensões finais:
Training: (29786, 2400)
Validation: (7447, 2400)
Redução de features: 21.9%

Treinando MLP com features reduzidas...

🏆 RESULTADOS - MÉTODO VARIÂNCIA:
Training Accuracy: 0.9750 (97.50%)
Validation Accuracy: 0.8203 (82.03%)
Overfitting Gap: 15.47%
Features removidas: 7 colunas [25, 26, 27, 28, 

## 11. Experimento: Método Manual (Contagem de Pixels)

In [ ]:
print("=== EXPERIMENTO 2: CROP MANUAL (CONTAGEM PIXELS) ===\n")
print("📝 Aguardando contagem de pixels do usuário...")
print("Quando você contar os pixels, altere o valor 'crop_col_start' abaixo:\n")

# Parâmetro que você vai ajustar após contar os pixels
crop_col_start = 26  # AJUSTE ESTE VALOR APÓS CONTAR OS PIXELS

# Testar método manual
X_train_manual, X_val_manual, y_train_manual, y_val_manual, class_weights_manual, scaler_manual, removed_cols_manual = enhanced_preprocessing_pipeline(
    X_train_full, y_train_full, method='manual', crop_param=crop_col_start
)

# Treinar MLP otimizado
print("\nTreinando MLP com crop manual...")
mlp_manual = MLPClassifier(
    hidden_layer_sizes=(1024, 256),
    random_state=42,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

mlp_manual.fit(X_train_manual, y_train_manual)

# Avaliar
train_acc_manual = mlp_manual.score(X_train_manual, y_train_manual)
val_acc_manual = mlp_manual.score(X_val_manual, y_val_manual)

print(f"\n🏆 RESULTADOS - MÉTODO MANUAL:")
print(f"Training Accuracy: {train_acc_manual:.4f} ({train_acc_manual*100:.2f}%)")
print(f"Validation Accuracy: {val_acc_manual:.4f} ({val_acc_manual*100:.2f}%)")
print(f"Overfitting Gap: {(train_acc_manual - val_acc_manual)*100:.2f}%")
print(f"Features removidas: {len(removed_cols_manual)} colunas {removed_cols_manual}")

## 12. Comparação Final dos Métodos

In [ ]:
print("=== COMPARAÇÃO FINAL DOS MÉTODOS ===\n")

# Baseline (do experimento anterior)
baseline_val_acc = classifier_results_baseline['Neural Network']['val_acc']
baseline_train_acc = classifier_results_baseline['Neural Network']['train_acc']

print(f"{'Método':<20} {'Train Acc':<12} {'Val Acc':<12} {'Gap':<8} {'Features':<10} {'Redução':<10}")
print("-" * 78)
print(f"{'Baseline':<20} {baseline_train_acc:<12.4f} {baseline_val_acc:<12.4f} {(baseline_train_acc-baseline_val_acc)*100:<8.2f} {3072:<10} {0:<10.1f}%")
print(f"{'Variância':<20} {train_acc_var:<12.4f} {val_acc_var:<12.4f} {(train_acc_var-val_acc_var)*100:<8.2f} {X_train_var.shape[1]:<10} {((3072-X_train_var.shape[1])/3072*100):<10.1f}%")
print(f"{'Manual':<20} {train_acc_manual:<12.4f} {val_acc_manual:<12.4f} {(train_acc_manual-val_acc_manual)*100:<8.2f} {X_train_manual.shape[1]:<10} {((3072-X_train_manual.shape[1])/3072*100):<10.1f}%")

# Melhor resultado
best_val = max(baseline_val_acc, val_acc_var, val_acc_manual)
if best_val == val_acc_var:
    best_method = "Variância"
elif best_val == val_acc_manual:
    best_method = "Manual"
else:
    best_method = "Baseline"

print(f"\n🏆 MELHOR MÉTODO: {best_method} com {best_val:.4f} ({best_val*100:.2f}%) de acurácia")

# Análise dos ganhos
if best_val > baseline_val_acc:
    improvement = (best_val - baseline_val_acc) * 100
    print(f"✅ Melhoria de {improvement:.2f} pontos percentuais sobre o baseline!")
else:
    print(f"❌ Métodos não superaram o baseline")

print(f"\n📈 INSIGHTS:")
print(f"  • Normalização per-channel {'ajudou' if best_val > baseline_val_acc else 'não ajudou'} na performance")
print(f"  • Redução de features {'reduziu overfitting' if (train_acc_var-val_acc_var) < (baseline_train_acc-baseline_val_acc) else 'não afetou overfitting'}")
print(f"  • {'Método de variância' if val_acc_var > val_acc_manual else 'Método manual'} foi mais eficaz para crop")

## 9. Preprocessing Inteligente baseado na Análise de Variância

In [ ]:
def per_channel_standardization(X):
    """
    Normaliza cada canal RGB individualmente para cada imagem.
    Remove viés de iluminação e melhora contraste local.
    """
    X_normalized = np.zeros_like(X, dtype=np.float32)
    
    for i in range(X.shape[0]):  # Para cada imagem
        for channel in range(3):  # Para cada canal RGB
            ch_data = X[i, :, :, channel]
            mean_ch = np.mean(ch_data)
            std_ch = np.std(ch_data)
            
            if std_ch > 1e-8:  # Evita divisão por zero
                X_normalized[i, :, :, channel] = (ch_data - mean_ch) / std_ch
            else:
                X_normalized[i, :, :, channel] = ch_data - mean_ch
    
    return X_normalized

def intelligent_crop_variance_based(X, variance_threshold=3.0):
    """
    Remove colunas com baixo poder discriminativo baseado na análise de variância da EDA.
    """
    print(f"Aplicando crop inteligente com threshold {variance_threshold}...")
    
    # Simular mapa de variância baseado na observação da EDA
    # Colunas à direita (26-31) têm baixa variância discriminativa
    
    if variance_threshold == 'visual':
        # Método baseado na observação visual da EDA
        X_cropped = X[:, :, :26, :]  # Manter primeiras 26 colunas
        removed_cols = list(range(26, 32))
    else:
        # Método heurístico: remover colunas da direita com baixa info
        # Baseado na análise que mostrou variância baixa nas bordas direitas
        crop_point = max(20, min(30, int(32 - (10 - variance_threshold))))
        X_cropped = X[:, :, :crop_point, :]
        removed_cols = list(range(crop_point, 32))
    
    print(f"Dimensões originais: {X.shape}")
    print(f"Dimensões após crop: {X_cropped.shape}")
    print(f"Colunas removidas: {len(removed_cols)} - {removed_cols}")
    
    return X_cropped, removed_cols

def manual_crop_by_pixel(X, crop_col_start=26):
    """
    Crop manual baseado na contagem de pixels do usuário.
    """
    print(f"Aplicando crop manual a partir da coluna {crop_col_start}...")
    
    X_cropped = X[:, :, :crop_col_start, :]
    removed_cols = list(range(crop_col_start, X.shape[2]))
    
    print(f"Dimensões originais: {X.shape}")
    print(f"Dimensões após crop: {X_cropped.shape}")
    print(f"Colunas removidas: {len(removed_cols)} - {removed_cols}")
    
    return X_cropped, removed_cols

def enhanced_preprocessing_pipeline(X_train_full, y_train_full, method='variance', crop_param=3.0):
    """
    Pipeline completo de preprocessing inteligente.
    
    Args:
        method: 'variance' ou 'manual'
        crop_param: threshold para variance ou coluna para manual
    """
    
    print(f"=== PREPROCESSING INTELIGENTE - MÉTODO {method.upper()} ===")
    
    # 1. Reshape para imagens e normalização básica
    X_images = X_train_full.reshape(-1, 32, 32, 3).astype('float32') / 255.0
    
    # 2. Crop inteligente
    if method == 'variance':
        X_cropped, removed_cols = intelligent_crop_variance_based(X_images, crop_param)
    else:  # manual
        X_cropped, removed_cols = manual_crop_by_pixel(X_images, int(crop_param))
    
    # 3. Normalização per-channel
    print("\nAplicando normalização per-channel...")
    X_normalized = per_channel_standardization(X_cropped)
    
    # 4. Split estratificado
    X_train, X_val, y_train, y_val = train_test_split(
        X_normalized, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )
    
    # 5. Flatten para ML tradicional
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    
    # 6. Scaling final
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_val_scaled = scaler.transform(X_val_flat)
    
    # 7. Calcular class weights
    class_weights = compute_class_weight(
        'balanced', classes=np.unique(y_train_full), y=y_train_full
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    print(f"\nDimensões finais:")
    print(f"Training: {X_train_scaled.shape}")
    print(f"Validation: {X_val_scaled.shape}")
    print(f"Redução de features: {((3072 - X_train_scaled.shape[1]) / 3072 * 100):.1f}%")
    
    return X_train_scaled, X_val_scaled, y_train, y_val, class_weight_dict, scaler, removed_cols

print("Funções de preprocessing inteligente carregadas!")